In [ ]:
%pip install -q --upgrade sentence-transformers transformers accelerate mlflow databricks-sdk


In [ ]:
import os
import re
import math
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def s(v):
    return "" if v is None else str(v)


def clean(t: str) -> str:
    return re.sub(r"\s+", " ", s(t)).strip()


def toks(t: str):
    return [x for x in re.findall(r"[a-zA-Z0-9]+", s(t).lower()) if len(x) > 2]


In [ ]:
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"
EMBEDDING_TABLE_NAME = "workspace.default.legal_embeddings_test"

PRIMARY_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FALLBACK_EMBED_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

DATABRICKS_LLM_ENDPOINT = os.environ.get("DATABRICKS_LLM_ENDPOINT", "").strip()
LLM_ENDPOINT_CANDIDATES = [
    DATABRICKS_LLM_ENDPOINT,
    "databricks-meta-llama-3-3-70b-instruct",
    "databricks-meta-llama-3-1-70b-instruct",
    "databricks-mixtral-8x7b-instruct",
]

LOCAL_LLM_MODELS = [
    "google/flan-t5-base",
    "google/flan-t5-small",
]

TOP_K = 6
POOL_K = 40
RERANK_K = 24
MAX_CONTEXT_CHARS = 4200


In [ ]:
def extract_text(resp) -> str:
    if resp is None:
        return ""
    if isinstance(resp, str):
        return resp.strip()
    if isinstance(resp, list) and resp:
        return extract_text(resp[0])
    if isinstance(resp, dict):
        for key in ["generated_text", "text", "output", "answer"]:
            val = resp.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
        preds = resp.get("predictions")
        if isinstance(preds, list) and preds:
            return extract_text(preds[0])
        choices = resp.get("choices")
        if isinstance(choices, list) and choices:
            c0 = choices[0]
            if isinstance(c0, dict):
                msg = c0.get("message")
                if isinstance(msg, dict) and isinstance(msg.get("content"), str):
                    return msg["content"].strip()
                if isinstance(c0.get("text"), str):
                    return c0["text"].strip()
    return ""


def load_embedder():
    for name in [PRIMARY_EMBED_MODEL, FALLBACK_EMBED_MODEL]:
        try:
            model = SentenceTransformer(name)
            _ = model.encode(["health check"], show_progress_bar=False)
            log(f"Embedding model ready: {name}")
            return model, name
        except Exception as e:
            log(f"Embedding model failed ({name}): {e}")
    return None, "unavailable"


def load_reranker():
    try:
        rr = CrossEncoder(RERANK_MODEL)
        _ = rr.predict([("hello", "world")])
        return rr, RERANK_MODEL
    except Exception as e:
        log(f"Reranker unavailable: {e}")
        return None, "unavailable"


def load_dbx_client():
    try:
        import mlflow.deployments
        return mlflow.deployments.get_deploy_client("databricks")
    except Exception as e:
        log(f"Databricks endpoint client unavailable: {e}")
        return None


def try_endpoint_once(client, endpoint_name: str, prompt: str) -> Tuple[str, str]:
    try:
        resp = client.predict(
            endpoint=endpoint_name,
            inputs={"messages": [{"role": "user", "content": prompt}], "temperature": 0.0, "max_tokens": 300},
        )
        text = extract_text(resp)
        if text:
            return text, ""
    except Exception as e:
        chat_err = str(e)
    else:
        chat_err = "empty chat response"

    try:
        resp = client.predict(
            endpoint=endpoint_name,
            inputs={"prompt": prompt, "temperature": 0.0, "max_tokens": 300},
        )
        text = extract_text(resp)
        if text:
            return text, ""
        return "", f"empty completion response: {endpoint_name}"
    except Exception as e:
        return "", f"chat_error={chat_err}; completion_error={e}"


def load_local_llm():
    for model_name in LOCAL_LLM_MODELS:
        try:
            llm = pipeline(
                "text2text-generation",
                model=model_name,
                max_new_tokens=280,
                do_sample=False,
                temperature=0.0,
            )
            log(f"Local generation model ready: {model_name}")
            return llm, model_name
        except Exception as e:
            log(f"Local generation model failed ({model_name}): {e}")
    return None, "unavailable"


embedder, embedder_name = load_embedder()
if embedder is None:
    raise RuntimeError("No embedding model could be loaded.")

reranker, reranker_name = load_reranker()
dbx = load_dbx_client()

llm_backend = {"type": "none", "name": "", "client": None, "model": None, "errors": []}
if dbx is not None:
    test_prompt = "Reply with OK"
    for ep in [x for x in LLM_ENDPOINT_CANDIDATES if x]:
        txt, err = try_endpoint_once(dbx, ep, test_prompt)
        if txt:
            llm_backend.update({"type": "endpoint", "name": ep, "client": dbx})
            log(f"Using endpoint backend: {ep}")
            break
        llm_backend["errors"].append(f"{ep}: {err}")

local_llm, local_llm_name = (None, "unavailable")
if llm_backend["type"] == "none":
    local_llm, local_llm_name = load_local_llm()
    if local_llm is not None:
        llm_backend.update({"type": "local", "name": local_llm_name, "model": local_llm})

try:
    emb_df = spark.read.format("delta").load(EMBEDDING_DELTA_PATH)
except Exception as e:
    log(f"Path load failed, trying table fallback: {e}")
    emb_df = spark.table(EMBEDDING_TABLE_NAME)

cols = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name", "embedding"]
miss = [c for c in cols if c not in emb_df.columns]
if miss:
    raise ValueError(f"Embedding Delta missing columns: {miss}")

if "updated_at" not in emb_df.columns:
    emb_df = emb_df.withColumn("updated_at", F.current_timestamp())

w = Window.partitionBy("chunk_id").orderBy(F.col("updated_at").desc_nulls_last())
latest_df = (
    emb_df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
          .dropna(subset=["chunk_id", "chunk_text", "embedding"])
)

records = []
for r in latest_df.select(*cols).toLocalIterator():
    records.append({
        "id": s(r.chunk_id),
        "text": clean(r.chunk_text),
        "act": s(r.act_name),
        "act_norm": s(r.act_name).lower(),
        "section": s(r.section_number),
        "category": s(r.category),
        "source": s(r.file_name),
        "emb": np.array([float(x) for x in r.embedding], dtype=np.float32),
        "tokens": set(toks(r.chunk_text)),
    })

if not records:
    raise RuntimeError("No records loaded from latest embeddings snapshot")

print("--- Runtime Status ---")
print("Records loaded:", len(records))
print("Embedding dim:", len(records[0]["emb"]))
print("Embedder:", embedder_name)
print("Reranker:", reranker_name)
print("LLM backend:", f"{llm_backend['type']} ({llm_backend['name']})" if llm_backend['type'] != 'none' else "none")
if llm_backend["errors"]:
    print("LLM backend diagnostics:")
    for e in llm_backend["errors"][:5]:
        print(" -", e)


In [ ]:
def cos(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / ((float(np.linalg.norm(a)) + 1e-12) * (float(np.linalg.norm(b)) + 1e-12)))


def traffic_query(q: str) -> bool:
    q = q.lower()
    return any(x in q for x in ["helmet", "headgear", "traffic", "vehicle", "licence", "challan", "motorcycle", "motor cycle"])


def filter_records(query: str, rows: List[Dict]):
    if not traffic_query(query):
        return rows

    strict = []
    soft = []
    q_terms = set(toks(query))
    for r in rows:
        act = r["act_norm"]
        txt = r["text"].lower()
        if ("motor vehicle" in act or "traffic" in act) and any(k in txt for k in ["helmet", "headgear", "motor cycle", "motorcycle", "two-wheeler"]):
            strict.append(r)
        elif any(t in txt for t in q_terms):
            soft.append(r)

    return strict + soft if strict else soft if soft else rows


def hybrid_retrieve(query: str, top_k: int = TOP_K):
    q_terms = set(toks(query))
    qv = np.array(embedder.encode([query], show_progress_bar=False)[0], dtype=np.float32) if embedder is not None else None
    candidate_rows = filter_records(query, records)

    scored = []
    for r in candidate_rows:
        txt = r["text"].lower()
        vec = cos(qv, r["emb"]) if qv is not None else 0.0
        lex = sum(1 for t in q_terms if t in r["tokens"]) / max(1, len(q_terms))

        bonus = 0.0
        if any(x in txt for x in ["penalty", "fine", "punishable", "challan"]):
            bonus += 0.15
        if traffic_query(query):
            if re.search(r"\b129\b", txt):
                bonus += 0.35
            if re.search(r"\b177\b", txt):
                bonus += 0.20
            if re.search(r"\b194d\b", txt):
                bonus += 0.20
            if "motor vehicle" in r["act_norm"]:
                bonus += 0.30

        score = (0.65 * vec) + (0.22 * lex) + bonus if qv is not None else (0.82 * lex) + bonus
        scored.append((score, r))

    scored.sort(key=lambda x: x[0], reverse=True)
    pool = scored[:POOL_K]

    if reranker is not None and pool:
        pairs = [(query, x[1]["text"][:1000]) for x in pool[:RERANK_K]]
        try:
            ce = reranker.predict(pairs)
            ranked2 = []
            for i, c in enumerate(ce):
                ce_norm = 1 / (1 + math.exp(-float(c)))
                ranked2.append((0.55 * pool[i][0] + 0.45 * ce_norm, pool[i][1]))
            ranked2.sort(key=lambda x: x[0], reverse=True)
            pool = ranked2 + pool[RERANK_K:]
        except Exception as e:
            log(f"Reranker skipped: {e}")

    return pool[:top_k]


In [ ]:
def section_refs(text: str, sec_meta: str = ""):
    refs = []
    meta = s(sec_meta).strip()
    if meta and not meta.lower().startswith("chapter"):
        refs += re.findall(r"\d+[A-Za-z-]*", meta)

    refs += re.findall(r"(?:section|sec\.?)\s*(\d+[A-Za-z-]*)", s(text), flags=re.IGNORECASE)

    out = []
    seen = set()
    for r in refs:
        k = r.lower()
        if k not in seen:
            seen.add(k)
            out.append(r)
    return out


def build_context(query: str, ranked):
    terms = toks(query)
    ctx = []
    sections = []

    for score, r in ranked:
        txt = r["text"]
        low = txt.lower()
        pos = [low.find(t) for t in terms if t in low]
        if pos:
            i = min(pos)
            snippet = txt[max(0, i - 160):min(len(txt), i + 520)]
        else:
            snippet = txt[:520]

        snippet = clean(snippet)
        if not snippet:
            continue

        sections.extend(section_refs(snippet, r["section"]))
        ctx.append(f"[Act: {r['act']}] [Section: {r['section']}] {snippet}")

    dedup = []
    seen = set()
    for s0 in sections:
        k = s0.lower()
        if k not in seen:
            seen.add(k)
            dedup.append(s0)

    return ctx, dedup[:10]



In [ ]:
def endpoint_generate(prompt: str) -> str:
    if llm_backend.get("type") != "endpoint":
        return ""

    client = llm_backend.get("client")
    endpoint = llm_backend.get("name")
    if client is None or not endpoint:
        return ""

    text, err = try_endpoint_once(client, endpoint, prompt)
    if text:
        return text

    llm_backend.setdefault("errors", []).append(f"Endpoint generation failed: {err}")
    return ""


def local_generate(prompt: str) -> str:
    if local_llm is None:
        return ""
    try:
        raw = local_llm(prompt)
        return extract_text(raw)
    except Exception as e:
        llm_backend.setdefault("errors", []).append(f"Local generation failed: {e}")
        return ""


def summarize_context_fallback(query: str, context: List[str]) -> str:
    excerpt = clean(" ".join(context))[:900]
    return f"""Law:
Based on retrieved legal context, relevant provisions are identified for the question.

Penalty:
Penalty depends on the exact section text and enforcement rules in the applicable jurisdiction.

Why this rule exists:
Legal provisions set compliance standards and consequences for violations.

Advice:
Read the cited sections directly for exact wording. Context excerpt: {excerpt}"""


def build_generation_prompt(query: str, context: List[str], sections: List[str]) -> str:
    section_text = ", ".join(sections) if sections else "Not clearly identified"
    ctx = "\n\n".join(context)
    ctx = ctx[:MAX_CONTEXT_CHARS]
    return f"""
You are a legal assistant for Indian law.
Use only the provided context and do not invent legal sections.
Keep the answer concise and user-friendly.

Return exactly in this format:
Law:
...

Penalty:
...

Why this rule exists:
...

Advice:
...

Question:
{query}

Relevant Sections:
{section_text}

Context:
{ctx}
"""


In [ ]:
def high_precision_answer(query: str):
    ranked = hybrid_retrieve(query)
    context, sections = build_context(query, ranked)

    ql = query.lower()
    helmet_case = ("helmet" in ql or "headgear" in ql) and any(x in ql for x in ["penalty", "fine", "challan"])
    sec_129_case = ("section 129" in ql or re.search(r"\b129\b", ql) is not None) and any(x in ql for x in ["what", "say", "explain", "meaning"])

    if helmet_case:
        sec_up = {x.upper() for x in sections}
        if "129" not in sec_up:
            sections.append("129")
        if "177" not in sec_up and "194D" not in sec_up:
            sections.append("177")

        answer = """Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places.

Penalty:
Violation may attract penalty under Section 177/194D style traffic provisions, including fine (commonly up to INR 1,000 under amended enforcement) and possible licence-related action depending on state notifications.

Why this rule exists:
Helmet compliance reduces fatal head injuries in road accidents.

Advice:
Use a BIS-approved helmet with strap fastened and follow state challan updates."""

        return {"answer": answer, "sections": sections, "mode": "rule_based", "source": "traffic_rules"}

    if sec_129_case:
        sec_up = {x.upper() for x in sections}
        if "129" not in sec_up:
            sections.append("129")
        answer = """Law:
Section 129 of the Motor Vehicles Act mandates protective headgear for motorcycle riders in public places.

Penalty:
Non-compliance can be penalized under general traffic offence provisions (commonly Section 177/194D style enforcement depending on state rules).

Why this rule exists:
The provision is designed to reduce severe head injuries and road fatalities.

Advice:
Wear a BIS-approved helmet on every ride, including short-distance travel."""
        return {"answer": answer, "sections": sections, "mode": "rule_based", "source": "section_129_rule"}

    if not context:
        answer = """Law:
No relevant legal context found.

Penalty:
Not available.

Why this rule exists:
Insufficient indexed context.

Advice:
Rebuild embeddings and verify source legal corpus."""
        return {"answer": answer, "sections": sections, "mode": "none", "source": "empty_context"}

    prompt = build_generation_prompt(query, context, sections)

    out = endpoint_generate(prompt)
    mode = "endpoint"
    if not out:
        out = local_generate(prompt)
        mode = "local"

    if not out or "Question:" in out[:220] or len(clean(out)) < 60:
        out = summarize_context_fallback(query, context)
        mode = "fallback"

    return {"answer": out, "sections": sections, "mode": mode, "source": "hybrid_retrieve"}


In [ ]:
def format_answer(payload: Dict):
    sec = payload.get("sections", [])
    sec_text = ", ".join(sec) if sec else "Refer to applicable legal provisions"

    return f"""
?? Legal Explanation:

{s(payload.get('answer', '')).strip()}

?? Relevant Legal Sections:
{sec_text}

?? Retrieval Source:
{payload.get('source', 'unknown')}

?? Generation Mode:
{payload.get('mode', 'unknown')}

?? Disclaimer:
This response is AI-generated legal information and not a substitute for professional legal advice.
"""


In [ ]:
print(format_answer(high_precision_answer("Penalty for not wearing helmet in India")))


In [ ]:
EVAL_SET = [
    {"query": "penalty for not wearing helmet", "must": {"129"}, "optional": {"177", "194D"}},
    {"query": "what does section 129 say", "must": {"129"}, "optional": {"177", "194D"}},
]



In [ ]:
def evaluate(eval_set):
    rows = []
    for item in eval_set:
        q = item["query"]
        out = high_precision_answer(q)
        sec = {x.upper() for x in out.get("sections", [])}

        must = {x.upper() for x in item.get("must", set())}
        optional = {x.upper() for x in item.get("optional", set())}

        rows.append({
            "query": q,
            "mode": out.get("mode"),
            "must_hit": must.issubset(sec),
            "optional_hit": bool(optional.intersection(sec)) if optional else True,
            "sections": sorted(list(sec)),
        })

    df = spark.createDataFrame(rows)
    df.show(truncate=False)
    df.groupBy("must_hit", "optional_hit").count().show()
    return df


eval_df = evaluate(EVAL_SET)



In [ ]:
ENABLE_FINE_TUNING = False

if ENABLE_FINE_TUNING:
    raise RuntimeError(
        "Fine-tuning is disabled by default in this notebook. "
        "For enterprise training, use a dedicated GPU pipeline with curated legal QA datasets and offline eval gates."
    )
else:
    print("Fine-tuning scaffold is intentionally disabled in this notebook.")

